# Generate final benchmark reports

This step is CPU-compatible and idempotent. It rebuilds report artifacts from measured evaluation JSON files in Drive.


In [ ]:
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

SMOKE_TEST = os.environ.get("SMOKE_TEST", "0").lower() in {"1", "true", "yes", "on"}
try:
    IS_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IS_COLAB = False
REPOSITORY_URL = os.environ.get(
    "BENCHMARK_REPOSITORY_URL",
    "https://github.com/Harryphan72007/aerial-object-detection-benchmark.git",
)
REPOSITORY_BRANCH = os.environ.get("BENCHMARK_REPOSITORY_BRANCH", "main")
if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/aerial-object-detection-benchmark")
    if not (REPO_DIR / ".git").is_dir():
        subprocess.run(
            ["git", "clone", "--branch", REPOSITORY_BRANCH, REPOSITORY_URL, str(REPO_DIR)],
            check=True,
        )
else:
    REPO_DIR = Path(os.environ.get("BENCHMARK_REPO_ROOT", Path.cwd())).resolve()
if not (REPO_DIR / "pyproject.toml").is_file():
    raise RuntimeError(f"Repository root is invalid: {REPO_DIR}")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
if IS_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-dataset-colab.txt"],
        check=True,
    )
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
DRIVE_ROOT = os.environ.get(
    "VISDRONE_DRIVE_ROOT",
    "/content/drive/MyDrive/visdrone_architecture_benchmark"
    if IS_COLAB
    else str(REPO_DIR / ".notebook-smoke"),
)
from src.paths import ProjectPaths
from src.reproducibility import seed_everything
from src.utils.environment import collect_environment
paths = ProjectPaths.from_value(DRIVE_ROOT).create()
seed_everything(42)
print({"repo": str(REPO_DIR), "storage": str(paths.root), "smoke_test": SMOKE_TEST})
collect_environment()


## Configuration


In [ ]:
MODEL_ID = "rtdetrv2_l"
DATASET_TRACK = "2class"


## Generate and verify reports


In [ ]:
from src.benchmark_status import discover_model_status
if SMOKE_TEST:
    print("SMOKE_TEST: report workflow imports passed; measured evaluation is not required.")
else:
    status = discover_model_status(DRIVE_ROOT, MODEL_ID, REPO_DIR)
    if status["evaluation_status"] != "COMPLETE":
        raise RuntimeError("Evaluation is missing. Run notebook 07 first.")
    subprocess.run(
        [sys.executable, "scripts/generate_report.py", "--drive-root", DRIVE_ROOT],
        check=True,
    )
    status = discover_model_status(DRIVE_ROOT, MODEL_ID, REPO_DIR)
    if status["report_status"] != "COMPLETE":
        raise RuntimeError("The generated report does not contain the selected final run.")
recommended = f"{MODEL_ID}__{DATASET_TRACK}__" + __import__("datetime").datetime.now(
    __import__("datetime").timezone.utc
).strftime("%Y%m%d_%H%M%S")
print("\nRESULTS READY FOR REVIEW")
print("\nEvaluation directory:", paths.evaluation)
print("Report directory:", paths.reports)
print("Recommended result bundle ID:", recommended)
print("Next notebook:", REPO_DIR / "notebooks" / "11_sync_results_to_github.ipynb")
